# Packages import

In [19]:
import os
import yaml
import requests
import pandas as pd
from requests.auth import HTTPBasicAuth
from bs4 import BeautifulSoup

## Apollo Scraper

In [20]:
url = "https://planzajec.uek.krakow.pl/index.php?typ=G&id=252671&okres=1"

In [21]:
with open("config.yaml", "r", encoding="utf-8") as yf:
    config = yaml.load(yf, Loader=yaml.SafeLoader)
username = config.get("credentials", {}).get("username") if config else None
password = config.get("credentials", {}).get("password") if config else None
auth = HTTPBasicAuth(username, password)

In [22]:
response = requests.get(url, auth=auth)
response.encoding = "UTF-8"
print(response.status_code)

200


In [23]:
page_dom = BeautifulSoup(response.text, "html.parser")

In [24]:
group = page_dom.select_one("div.grupa").get_text()
print(group)

ZICSS1-1212


In [25]:
classes_tag = page_dom.select_one("table")
with open("temp.html", "w", encoding="UTF-8") as hf:
    hf.write(str(classes_tag))

classes = pd.read_html("temp.html", encoding="UTF-8")[0]
os.remove("temp.html")

In [26]:
classes = classes.loc[classes["Typ"].isin(["ćwiczenia", "wykład", "egzamin"])]

In [27]:
classes[["Day", "Start time", "hyphen", "End time", "Duration"]] = classes["Dzień, godzina"].str.split(" ", expand=True)

In [ ]:
classes["Duration"] = classes["Duration"].map(
    lambda x: x.split("(")[1].split("g")[0] if isinstance(x, str) and "(" in x else x
)
classes
classes

AttributeError: 'float' object has no attribute 'split'

In [8]:
if not os.path.exists("./schedules"):
    os.mkdir("./schedules")

In [9]:
classes.to_csv(f"schedules/{group}.csv", encoding="UTF-8")